# LLM Eval - Testing Models & Store

This notebook tests the `models.py` and `store.py` modules.

In [ ]:
# Add the package to path (if not installed)
import sys
sys.path.insert(0, '.')

In [ ]:
from llm_eval.models import Prompt, Trace, Annotation
from llm_eval.store import TraceStore
import hashlib
from uuid import uuid4

## 1. Initialize Store

Creates a local SQLite database file `test_traces.db` in this directory.

In [ ]:
# This creates test_traces.db in the current directory
store = TraceStore("test_traces.db")
print("Store initialized! Database file: test_traces.db")

## 2. Test Prompt Storage

In [ ]:
# Create a prompt using the Pydantic model
template = "You are a helpful assistant. Summarize the following: {text}"
template_hash = hashlib.sha256(template.encode()).hexdigest()

prompt = Prompt(
    name="summarizer",
    version=1,
    template=template,
    description="Initial summarizer prompt",
    template_hash=template_hash
)

print("Prompt model created:")
print(prompt.model_dump_json(indent=2))

In [ ]:
# Save to database
prompt_id = store.save_prompt(prompt.model_dump())
print(f"Saved prompt with ID: {prompt_id}")

In [ ]:
# Retrieve it back
retrieved = store.get_prompt("summarizer")
print("Retrieved prompt:")
print(retrieved)

In [ ]:
# Save version 2
template_v2 = "You are an expert summarizer. Create a concise summary of: {text}"
prompt_v2 = Prompt(
    name="summarizer",
    version=2,
    template=template_v2,
    description="Improved prompt with 'expert' framing",
    template_hash=hashlib.sha256(template_v2.encode()).hexdigest()
)
store.save_prompt(prompt_v2.model_dump())

# List all versions
versions = store.list_prompt_versions("summarizer")
print(f"Found {len(versions)} versions:")
for v in versions:
    print(f"  v{v['version']}: {v['description']}")

## 3. Test Trace Storage

In [ ]:
# Create a trace using the Pydantic model
session_id = str(uuid4())

trace = Trace(
    project="test_project",
    session_id=session_id,
    prompt_name="summarizer",
    prompt_version=1,
    input_messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Summarize: The quick brown fox..."}
    ],
    output_content="A fox jumped over a dog.",
    input_tokens=25,
    output_tokens=8,
    total_tokens=33,
    model_name="gpt-4",
    latency_ms=1234,
    metadata={"temperature": 0.7},
    status="success"
)

print("Trace model created:")
print(f"  ID: {trace.id}")
print(f"  Session: {trace.session_id}")

In [ ]:
# Save trace
trace_id = store.save_trace(trace.model_dump())
print(f"Saved trace with ID: {trace_id}")

In [ ]:
# Add another trace to the same session
trace2 = Trace(
    project="test_project",
    session_id=session_id,  # Same session
    prompt_name="summarizer",
    prompt_version=2,  # Using v2 now
    input_messages=[
        {"role": "user", "content": "Summarize: Lorem ipsum dolor sit amet..."}
    ],
    output_content="Latin placeholder text.",
    input_tokens=15,
    output_tokens=5,
    total_tokens=20,
    model_name="gpt-4",
    latency_ms=890,
    status="success"
)
store.save_trace(trace2.model_dump())
print(f"Saved second trace: {trace2.id}")

In [ ]:
# Retrieve trace
retrieved_trace = store.get_trace(trace_id)
print("Retrieved trace:")
print(f"  Output: {retrieved_trace['output_content']}")
print(f"  Tokens: {retrieved_trace['total_tokens']}")
print(f"  Input messages: {retrieved_trace['input_messages']}")

In [ ]:
# Get all traces for the session
session_traces = store.get_traces({"session_id": session_id})
print(f"Found {len(session_traces)} traces in session:")
for t in session_traces:
    print(f"  - {t['id'][:8]}... | v{t['prompt_version']} | {t['total_tokens']} tokens")

## 4. Test Session Summaries

In [ ]:
# Get session summaries
sessions = store.get_sessions()
print("Sessions:")
for s in sessions:
    print(f"  {s['session_id'][:8]}... | {s['trace_count']} traces | {s['total_tokens']} tokens")

## 5. Test Annotations

In [ ]:
# Create an annotation
annotation = Annotation(
    trace_id=trace_id,
    rating="good",
    notes="Accurate summary, good conciseness",
    failure_category="",
    annotator="test_user"
)

ann_id = store.save_annotation(annotation.model_dump())
print(f"Saved annotation with ID: {ann_id}")

In [ ]:
# Retrieve annotation
retrieved_ann = store.get_annotation(trace_id)
print("Retrieved annotation:")
print(f"  Rating: {retrieved_ann['rating']}")
print(f"  Notes: {retrieved_ann['notes']}")
print(f"  Annotator: {retrieved_ann['annotator']}")

## 6. Verify Database File

In [ ]:
import os
db_path = "test_traces.db"
if os.path.exists(db_path):
    size = os.path.getsize(db_path)
    print(f"Database file exists: {db_path}")
    print(f"Size: {size:,} bytes")
else:
    print("Database file not found!")

## Cleanup (optional)

In [ ]:
# Uncomment to delete test database
# import os
# os.remove("test_traces.db")
# print("Deleted test_traces.db")